# Steam Narrative Viz — Data Cleaning
Primary dataset: `93182_steam_games.csv` (93,182 rows × 39 columns)

## Step 1: Load Data and Initial Inspection

In [16]:
import pandas as pd
import numpy as np

df = pd.read_csv('data/93182_steam_games.csv', low_memory=False)
print(f'Shape: {df.shape}')
df.head(3)

Shape: (93182, 39)


,AppID,Name,Release date,Estimated owners,Peak CCU,Required age,Price,DLC count,About the game,Supported languages,...,Average playtime two weeks,Median playtime forever,Median playtime two weeks,Developers,Publishers,Categories,Genres,Tags,Screenshots,Movies
0,1424640,余烬,"Oct 3, 2020",20000 - 50000,0,0,3.99,0,'Ashes of war' is an anti war theme adventure ...,['Simplified Chinese'],...,0,0,0,宁夏华夏西部影视城有限公司,宁夏华夏西部影视城有限公司,"Single-player,Family Sharing","Adventure,Casual,Indie,RPG","Sokoban,RPG,Puzzle-Platformer,Exploration,Adve...",https://shared.akamai.steamstatic.com/store_it...,http://video.akamai.steamstatic.com/store_trai...
1,402890,Nyctophilia,"Sep 23, 2015",50000 - 100000,0,0,0.00,0,NYCTOPHILIA Nyctophilia is an 2D psychological...,"['English', 'Russian']",...,0,0,0,Cat In A Jar Games,Cat In A Jar Games,Single-player,"Adventure,Free To Play,Indie","Free to Play,Indie,Adventure,Horror,2D,Pixel G...",https://shared.akamai.steamstatic.com/store_it...,http://video.akamai.steamstatic.com/store_trai...
2,1151740,Prison Princess,"Apr 2, 2020",0 - 20000,0,0,19.99,0,"ABOUT Now nothing more than a phantom, can the...","['English', 'Simplified Chinese', 'Traditional...",...,0,0,0,qureate,qureate,"Single-player,Steam Achievements,Full controll...","Adventure,Indie","Sexual Content,Adventure,Indie,Nudity,Anime,Ma...",https://shared.akamai.steamstatic.com/store_it...,http://video.akamai.steamstatic.com/store_trai...


In [17]:
# Missing value summary by column
missing = df.isnull().sum()
missing_pct = (missing / len(df) * 100).round(1)
pd.DataFrame({'missing_count': missing, 'missing_pct': missing_pct}).query('missing_count > 0').sort_values('missing_pct', ascending=False)

,missing_count,missing_pct
Score rank,93175,100.0
Metacritic url,89159,95.7
Reviews,82583,88.6
Tags,80264,86.1
Notes,77843,83.5
Estimated owners,76720,82.3
Website,51528,55.3
Support url,48954,52.5
Support email,15834,17.0
Movies,7647,8.2


## Step 2: Filter Non-Game Entries (DLC, Software, Soundtracks, etc.)

In [18]:
# Check the most frequent genre labels (top 20)
genre_counts = df['Genres'].dropna().str.split(',').explode().str.strip().value_counts()
print(genre_counts.head(20))

Genres
Indie                    61959
Casual                   37819
Action                   36689
Adventure                34866
Simulation               17962
Strategy                 17218
RPG                      16058
Early Access              9084
Free To Play              8061
Sports                    3928
Racing                    3240
Massively Multiplayer     2152
Utilities                  855
Design & Illustration      504
Violent                    436
Education                  401
Animation & Modeling       400
Video Production           295
Gore                       268
Game Development           258
Name: count, dtype: int64


In [19]:
# Keep rows that look like games based on genre tags
def is_game(genre_str):
    if pd.isna(genre_str):
        return False
    genres = {g.strip().lower() for g in genre_str.split(',')}
    # Keep entries with at least one typical game genre
    game_genres = {'action', 'adventure', 'casual', 'indie', 'massively multiplayer',
                   'racing', 'rpg', 'simulation', 'sports', 'strategy', 'free to play'}
    return bool(genres & game_genres)

df_games = df[df['Genres'].apply(is_game)].copy()
print(f'Rows after filtering: {len(df_games)} (original: {len(df)}, removed: {len(df) - len(df_games)})')

Rows after filtering: 87098 (original: 93182, removed: 6084)


## Step 3: Parse Release Year

In [20]:
# Check the format of the Release date field
print(df_games['Release date'].dropna().head(10).tolist())
print('\nMissing values:', df_games['Release date'].isna().sum())

['Oct 3, 2020', 'Sep 23, 2015', 'Apr 2, 2020', 'Oct 12, 2018', 'Mar 11, 2022', 'Feb 11, 2016', 'Apr 4, 2019', 'Feb 20, 2024', 'Dec 3, 2020', 'Mar 10, 2017']

Missing values: 0


In [21]:
df_games['release_year'] = pd.to_datetime(
    df_games['Release date'], errors='coerce'
).dt.year

# Remove rows with out-of-range years (keep only 2003-2024)
df_games = df_games[(df_games['release_year'] >= 2003) & (df_games['release_year'] <= 2024)]

print('Year distribution:')
print(df_games['release_year'].value_counts().sort_index())

Year distribution:
release_year
2003.0        3
2004.0        6
2005.0        7
2006.0       69
2007.0       93
2008.0      158
2009.0      314
2010.0      275
2011.0      275
2012.0      322
2013.0      455
2014.0     1490
2015.0     2464
2016.0     4040
2017.0     5763
2018.0     7343
2019.0     7132
2020.0     8707
2021.0    10410
2022.0    11495
2023.0    13702
2024.0    12452
Name: count, dtype: int64


## Step 4: Compute Review Metrics

In [22]:
df_games['Positive'] = pd.to_numeric(df_games['Positive'], errors='coerce').fillna(0)
df_games['Negative'] = pd.to_numeric(df_games['Negative'], errors='coerce').fillna(0)

df_games['total_reviews'] = df_games['Positive'] + df_games['Negative']
df_games['positive_pct'] = np.where(
    df_games['total_reviews'] > 0,
    df_games['Positive'] / df_games['total_reviews'] * 100,
    np.nan
)

print('total_reviews summary statistics:')
print(df_games['total_reviews'].describe())
print(f'\nGames with zero reviews: {(df_games["total_reviews"] == 0).sum()} ({(df_games["total_reviews"] == 0).mean()*100:.1f}%)')

total_reviews summary statistics:
count     86975.000000
mean        226.998505
std        6574.379629
min           0.000000
25%           0.000000
50%           0.000000
75%           0.000000
max      751820.000000
Name: total_reviews, dtype: float64

Games with zero reviews: 74314 (85.4%)


## Step 5: Parse Estimated Owners (Range String to Midpoint Value)

In [23]:
# Inspect raw format examples
print(df_games['Estimated owners'].dropna().unique()[:15])

['20000 - 50000' '50000 - 100000' '0 - 20000' '100000 - 200000' '0 - 0'
 '500000 - 1000000' '1000000 - 2000000' '5000000 - 10000000'
 '200000 - 500000' '2000000 - 5000000' '20000000 - 50000000'
 '10000000 - 20000000']


In [24]:
def parse_owners(s):
    if pd.isna(s):
        return np.nan
    parts = str(s).replace(',', '').replace('–', '-').replace('—', '-').split('-')
    parts = [p.strip() for p in parts]
    try:
        lo, hi = int(parts[0]), int(parts[1])
        return (lo + hi) / 2
    except (ValueError, IndexError):
        return np.nan

df_games['owners_midpoint'] = df_games['Estimated owners'].apply(parse_owners)
print('owners_midpoint summary statistics:')
print(df_games['owners_midpoint'].describe())

owners_midpoint summary statistics:
count    1.538600e+04
mean     8.765989e+04
std      6.990746e+05
min      0.000000e+00
25%      1.000000e+04
50%      1.000000e+04
75%      3.500000e+04
max      3.500000e+07
Name: owners_midpoint, dtype: float64


## Step 6: Flag Indie Games

In [25]:
df_games['is_indie'] = df_games['Genres'].str.contains('Indie', case=False, na=False)

print(f'Indie games: {df_games["is_indie"].sum()} ({df_games["is_indie"].mean()*100:.1f}%)')
print(f'Non-Indie:   {(~df_games["is_indie"]).sum()}')

Indie games: 61883 (71.2%)
Non-Indie:   25092


## Step 7: Quick Checks for Narrative Feasibility

In [26]:
# Scene 1: Annual release volume (is growth accelerating?)
yearly = df_games.groupby('release_year').size().reset_index(name='game_count')
print('Games released per year:')
print(yearly.to_string(index=False))

Games released per year:
 release_year  game_count
       2003.0           3
       2004.0           6
       2005.0           7
       2006.0          69
       2007.0          93
       2008.0         158
       2009.0         314
       2010.0         275
       2011.0         275
       2012.0         322
       2013.0         455
       2014.0        1490
       2015.0        2464
       2016.0        4040
       2017.0        5763
       2018.0        7343
       2019.0        7132
       2020.0        8707
       2021.0       10410
       2022.0       11495
       2023.0       13702
       2024.0       12452


In [27]:
# Scene 2: Review count distribution (is it heavy-tailed?)
buckets = [0, 1, 10, 100, 1000, 10000, 100000, float('inf')]
labels = ['0', '1-9', '10-99', '100-999', '1K-9999', '10K-99K', '100K+']
df_games['review_bucket'] = pd.cut(df_games['total_reviews'], bins=buckets, labels=labels, right=False)
dist = df_games['review_bucket'].value_counts().sort_index()
dist_pct = (dist / len(df_games) * 100).round(1)
print('Review count distribution:')
print(pd.DataFrame({'count': dist, 'pct': dist_pct}))

Review count distribution:
               count   pct
review_bucket             
0              74314  85.4
1-9             3974   4.6
10-99           5102   5.9
100-999         2461   2.8
1K-9999          865   1.0
10K-99K          223   0.3
100K+             36   0.0


In [28]:
# Scene 3: Indie vs Non-Indie yearly success comparison
cohort = df_games.groupby(['release_year', 'is_indie']).agg(
    game_count=('AppID', 'count'),
    median_reviews=('total_reviews', 'median'),
    pct_zero_reviews=('total_reviews', lambda x: (x == 0).mean() * 100),
    median_positive_pct=('positive_pct', 'median')
).reset_index()

print('Indie vs Non-Indie (recent 5-year sample):')
print(cohort[cohort['release_year'] >= 2019].to_string(index=False))

Indie vs Non-Indie (recent 5-year sample):
 release_year  is_indie  game_count  median_reviews  pct_zero_reviews  median_positive_pct
       2019.0     False        1584             0.0         84.911616            80.000000
       2019.0      True        5548             0.0         83.579668            79.207921
       2020.0     False        2100             0.0         85.761905            81.818182
       2020.0      True        6607             0.0         83.441804            82.920237
       2021.0     False        2968             0.0         85.478437            81.250000
       2021.0      True        7442             0.0         85.716205            84.090909
       2022.0     False        3538             0.0         87.733183            83.549784
       2022.0      True        7957             0.0         86.829207            85.714286
       2023.0     False        4388             0.0         87.944394            85.570639
       2023.0      True        9314            

## Step 8: Export Cleaned Dataset

In [29]:
cols_to_keep = [
    'AppID', 'Name', 'Release date', 'release_year',
    'Positive', 'Negative', 'total_reviews', 'positive_pct',
    'Estimated owners', 'owners_midpoint', 'Peak CCU',
    'Price', 'Genres', 'Tags', 'Developers', 'Publishers',
    'Average playtime forever', 'Median playtime forever',
    'Metacritic score', 'Recommendations',
    'is_indie', 'review_bucket'
]

df_clean = df_games[cols_to_keep].copy()
df_clean.to_csv('data/steam_clean.csv', index=False)

print(f'Saved: data/steam_clean.csv')
print(f'Final row count: {len(df_clean)}')
df_clean.head(3)

Saved: data/steam_clean.csv
Final row count: 86975


,AppID,Name,Release date,release_year,Positive,Negative,total_reviews,positive_pct,Estimated owners,owners_midpoint,...,Genres,Tags,Developers,Publishers,Average playtime forever,Median playtime forever,Metacritic score,Recommendations,is_indie,review_bucket
0,1424640,余烬,"Oct 3, 2020",2020.0,5,7,12,41.666667,20000 - 50000,35000.0,...,"Adventure,Casual,Indie,RPG","Sokoban,RPG,Puzzle-Platformer,Exploration,Adve...",宁夏华夏西部影视城有限公司,宁夏华夏西部影视城有限公司,0,0,0,0,True,10-99
1,402890,Nyctophilia,"Sep 23, 2015",2015.0,196,106,302,64.900662,50000 - 100000,75000.0,...,"Adventure,Free To Play,Indie","Free to Play,Indie,Adventure,Horror,2D,Pixel G...",Cat In A Jar Games,Cat In A Jar Games,0,0,0,0,True,100-999
2,1151740,Prison Princess,"Apr 2, 2020",2020.0,264,46,310,85.161290,0 - 20000,10000.0,...,"Adventure,Indie","Sexual Content,Adventure,Indie,Nudity,Anime,Ma...",qureate,qureate,0,0,0,299,True,100-999
